# Experimento — MLP para resolver XOR

**Aula 2 · 18/11 · Inteligência Artificial aplicada à Engenharia Elétrica**

## Antes de começar

### Objetivo
Aprender a tabela XOR usando rede 2–4–1. Entradas, ativação e perda são adimensionais.

### Pré-requisitos
Nenhum notebook anterior. Execute este arquivo de forma independente em uma sessão nova do Google Colab.

### Como executar
1. Abra o link público deste experimento no Colab: [Abrir MLP para resolver XOR](https://colab.research.google.com/github/Furriel/code-library/blob/main/cursos/ia/redes-neurais/mlp-xor.ipynb).
2. No menu **Ambiente de execução → Executar tudo**, rode as células de cima para baixo.
3. Faça a conta à mão antes de verificar a célula numérica.
4. Altere **somente o parâmetro sinalizado** no experimento guiado e execute tudo novamente.
5. Compare os gráficos e registre as conclusões; você não precisa enviar o notebook nem utilizar GitHub.

### Dados e parâmetros
O cálculo da propagação e do gradiente, a perda BCE, a superfície de decisão e o efeito de alterar apenas η.

O ambiente usa NumPy e Matplotlib disponíveis no Colab. O compilador C é opcional onde houver exemplo de referência.


In [ ]:
import json, math, shutil, subprocess
import numpy as np
import matplotlib.pyplot as plt

GCC = shutil.which("gcc") is not None
print("GCC disponível:", GCC)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

class LCG:
    # Mesmo gerador da página: s <- (1664525*s + 1013904223) mod 2^32; u = s / 2^32.
    def __init__(self, semente):
        self.s = semente % 2**32
    def proximo(self):
        self.s = (self.s * 1664525 + 1013904223) % 2**32
        return self.s / 4294967296
    def inteiro(self, n):
        return int(self.proximo() * n)

## Cálculo manual — propagação e retropropagação

Antes da rede 2–4–1 completa, faça um passo ilustrativo em uma MLP **2–1–1**, para que as contas caibam no papel. Considere `x=(1,0)`, pesos e vieses inicialmente nulos, alvo `d=1` e taxa `η=0,2`.

1. Camada oculta: `z_h=0`, `a=σ(0)=0,5`.
2. Saída: `z_o=0`, `y=σ(0)=0,5`. A entropia cruzada é `−ln(0,5)≈0,6931` (adimensional).
3. Gradiente de saída: `δ_o=y−d=−0,5`. Como a atualização usa o peso **anterior**, `w_o'=0−0,2×(−0,5)×0,5=0,05` e `b_o'=0,1`.
4. Gradiente da camada oculta: `δ_h=δ_o×w_o×a(1−a)=0` (o peso de saída anterior era zero); os pesos e o viés ocultos não mudam neste passo.

A rede usada no restante do notebook possui quatro neurônios ocultos; este exemplo pequeno ilustra a **mesma regra da cadeia**, não substitui a implementação validada em C.


In [ ]:
# Conferência da conta manual em uma MLP ilustrativa 2–1–1 (não altera a rede 2–4–1)
x=(1,0); d=1; eta=.2; w_saida=0.0; b_saida=0.0
ativacao=1/(1+math.exp(0))
y=1/(1+math.exp(0))
perda=-math.log(y)
delta_saida=y-d
delta_oculta=delta_saida*w_saida*ativacao*(1-ativacao)
w_saida_novo=w_saida-eta*delta_saida*ativacao
b_saida_novo=b_saida-eta*delta_saida
assert abs(ativacao-.5)<1e-12 and abs(y-.5)<1e-12
assert abs(perda-math.log(2))<1e-12 and abs(delta_oculta)<1e-12
assert abs(w_saida_novo-.05)<1e-12 and abs(b_saida_novo-.1)<1e-12
print(f"CÁLCULO MANUAL CONFERIDO: a={ativacao}; y={y}; BCE={perda:.4f}; w_o'={w_saida_novo:.2f}; b_o'={b_saida_novo:.2f}")


## 1. MLP 2–4–1: equações da retropropagação

$$a_j=\sigma\Big(b_{h,j}+\sum_i w_{h,ji}x_i\Big),\quad y=\sigma\Big(b_o+\sum_j w_{o,j}a_j\Big),\quad \sigma(z)=\frac{1}{1+e^{-z}}$$

$$L=-[d\ln y+(1-d)\ln(1-y)],\qquad \delta_o=\frac{\partial L}{\partial z_o}=y-d,\qquad \delta_{h,j}=\delta_o\,w_{o,j}\,a_j(1-a_j)$$

$$\theta \leftarrow \theta-\eta\,\frac{\partial L}{\partial\theta}\quad\text{(treino online, amostra por amostra)}$$

Os deltas são calculados **antes** de alterar os pesos. Acertar os quatro exemplos de treino não prova generalização.

In [ ]:
%%writefile mlp_xor.c
/*
 * mlp_xor.c — rede multicamada 2-4-1 que aprende XOR por retropropagação (C11 + math.h)
 * --------------------------------------------------------------------------------------
 * Projeto 1 — Inteligência Artificial aplicada à Engenharia Elétrica
 *
 * POR QUE ESTA REDE: um perceptron traça uma única reta e não separa XOR
 * ((0,1) e (1,0) de um lado; (0,0) e (1,1) do outro). Com 4 neurônios ocultos,
 * a rede combina várias fronteiras curvas e consegue.
 *
 * ARQUITETURA: 2 entradas -> 4 neurônios ocultos (sigmoide) -> 1 saída (sigmoide)
 *
 * EQUAÇÕES
 *     a_j = sig(bh_j + wh_j0*x0 + wh_j1*x1)       ativação do oculto j
 *     y   = sig(bo + soma_j wo_j*a_j)             saída (interpretada como "classe 1")
 *     sig(z) = 1 / (1 + exp(-z))
 *     L = -[d*ln(y) + (1-d)*ln(1-y)]              perda: entropia cruzada
 *     delta_saida   = y - d                       derivada de L em relação a z da saída
 *     delta_oculta_j = delta_saida * wo_j * a_j * (1 - a_j)   (regra da cadeia)
 *     peso <- peso - taxa * derivada              gradiente descendente
 *
 * ATENÇÃO: todos os deltas são calculados ANTES de atualizar os pesos de
 * saída; caso contrário delta_oculta usaria wo_j já modificado (errado).
 *
 * A página reproduz este programa em JavaScript (mesma semente e ordem) e um
 * teste automático confere: os dois param na época 400 com as mesmas saídas.
 *
 * Compilar: gcc -std=c11 -Wall -Wextra -Werror -pedantic -O2 mlp_xor.c -lm -o mlp_xor
 * (-lm liga a biblioteca matemática, necessária para exp e log)
 */
#include <stdio.h>
#include <math.h>
#include <stdint.h>

#define H 4              /* neurônios ocultos */
#define AMOSTRAS 4       /* linhas da tabela XOR */
#define MAX_EPOCAS 20000 /* limite de segurança */

/* Todos os 17 parâmetros da rede. */
typedef struct {
    double wh[H][2], bh[H], wo[H], bo;
} MLP;

/* Gerador pseudoaleatório (LCG) com semente fixa: a mesma rede inicial a cada
 * execução. Usa os 24 bits mais altos para obter um número em [0, 1). */
static uint32_t estado=11235813u;
static double rand01(void) {
    estado = estado*1664525u+1013904223u;
    return (double)(estado >> 8) / 16777216.0;
}
/* Peso inicial em [-1, 1). */
static double random_weight(void) { return 2.0*rand01()-1.0; }

/* Sigmoide em duas formas equivalentes para evitar estouro de exp() quando z
 * é muito negativo (estabilidade numérica). */
static double sigmoid(double z) {
    if(z>=0.0) return 1.0/(1.0+exp(-z));
    const double ez=exp(z);
    return ez/(1.0+ez);
}
/* Sorteia os pesos sempre na mesma ordem: wh[j][0], wh[j][1], bh[j], wo[j]; por fim bo. */
static void inicializar(MLP *rede) {
    for(int j=0;j<H;j++) {
        for(int i=0;i<2;i++) rede->wh[j][i]=random_weight();
        rede->bh[j]=random_weight();
        rede->wo[j]=random_weight();
    }
    rede->bo=random_weight();
}
/* Propagação direta. Preenche a[] com as ativações ocultas e devolve y. */
static double prever(const MLP *rede,const double x[2],double a[H]) {
    for(int j=0;j<H;j++) {
        double z=rede->bh[j];
        for(int i=0;i<2;i++) z+=rede->wh[j][i]*x[i];
        a[j]=sigmoid(z);
    }
    double z=rede->bo;
    for(int j=0;j<H;j++) z+=rede->wo[j]*a[j];
    return sigmoid(z);
}
/* Perda média (entropia cruzada) nos 4 exemplos. */
static double erro_medio(const MLP *rede,
                         const double x[AMOSTRAS][2],
                         const int d[AMOSTRAS]) {
    double custo=0.0;
    for(int k=0;k<AMOSTRAS;k++) {
        double a[H];double y=prever(rede,x[k],a);
        /* evita log(0) em casos extremos */
        if(y<1e-12) y=1e-12;
        if(y>1.0-1e-12) y=1.0-1e-12;
        custo-=d[k]*log(y)+(1-d[k])*log(1.0-y);
    }
    return custo/AMOSTRAS;
}
/* Um passo de treino com UMA amostra: propaga, retropropaga e atualiza. */
static void passo(MLP *rede,const double x[2],int d,double taxa) {
    double a[H];const double y=prever(rede,x,a);
    /* Sigmoide na saída + entropia cruzada: a derivada simplifica para y - d. */
    const double delta_saida=y-(double)d;
    double delta_oculta[H];
    /* Calcula todos os deltas antes de atualizar pesos de saida. */
    for(int j=0;j<H;j++)
        delta_oculta[j]=delta_saida*rede->wo[j]*a[j]*(1.0-a[j]);
    /* Gradiente descendente na camada de saída... */
    for(int j=0;j<H;j++)rede->wo[j]-=taxa*delta_saida*a[j];
    rede->bo-=taxa*delta_saida;
    /* ...e na camada oculta. */
    for(int j=0;j<H;j++) {
        for(int i=0;i<2;i++)rede->wh[j][i]-=taxa*delta_oculta[j]*x[i];
        rede->bh[j]-=taxa*delta_oculta[j];
    }
}
int main(void) {
    const double x[AMOSTRAS][2]={{0,0},{0,1},{1,0},{1,1}};
    const int d[AMOSTRAS]={0,1,1,0};                    /* tabela XOR */
    MLP rede={0};inicializar(&rede);
    int epoca_parada=-1;
    for(int e=1;e<=MAX_EPOCAS;e++) {
        /* treino "online": atualiza depois de cada amostra, taxa 0,7 */
        for(int k=0;k<AMOSTRAS;k++)passo(&rede,x[k],d[k],0.7);
        /* verifica a perda na 1ª época e depois a cada 100 épocas */
        if(e%100==0 || e==1) {
            const double custo=erro_medio(&rede,x,d);
            if(e%1000==0 || e==1)printf("epoca=%d bce=%.5f\n",e,custo);
            if(custo<0.04) {epoca_parada=e;break;}      /* critério de parada */
        }
    }
    printf("Convergiu (criterio BCE<0.04) na epoca=%d\n",epoca_parada);
    /* Teste nos 4 exemplos (os mesmos do treino: não prova generalização). */
    int corretos=0;
    for(int k=0;k<AMOSTRAS;k++) {
        double a[H],y=prever(&rede,x[k],a);
        const int classe=y>=0.5;                        /* limiar 0,5 */
        printf("x=(%.0f,%.0f) prob=%.4f previsto=%d esperado=%d\n",
               x[k][0],x[k][1],y,classe,d[k]);
        corretos+=(classe==d[k]);
    }
    printf("ACERTOS XOR: %d/4\n",corretos);
    return (epoca_parada>0 && corretos==AMOSTRAS)?0:1;  /* 0 = sucesso */
}


In [ ]:
if GCC:
    comp = subprocess.run(["gcc", "-std=c11", "-Wall", "-Wextra", "-Werror", "-pedantic", "-O2", "mlp_xor.c", "-lm", "-o", "mlp_xor"],
                          capture_output=True, text=True)
    print(comp.stdout + comp.stderr)
    assert comp.returncode == 0
    exec_mlp = subprocess.run(["./mlp_xor"], capture_output=True, text=True)
    print(exec_mlp.stdout)
    assert exec_mlp.returncode == 0 and "ACERTOS XOR: 4/4" in exec_mlp.stdout
else:
    exec_mlp = None
    print("GCC ausente: execução do C pulada.")

## 2. A mesma MLP em Python: perda, ativações, gradientes e superfície de decisão

A implementação usa o mesmo gerador (`rand01` do C), a mesma ordem de inicialização e a mesma função exponencial da biblioteca matemática; por isso a época de parada e as saídas coincidem com o C. Uma verificação por diferenças finitas confirma os gradientes.

In [ ]:
XOR = [((0, 0), 0), ((0, 1), 1), ((1, 0), 1), ((1, 1), 0)]
H = 4

def criar_mlp(semente=11235813):
    s = semente
    def r01():
        nonlocal s
        s = (s * 1664525 + 1013904223) % 2**32
        return (s >> 8) / 16777216.0
    rede = {"wh": [], "bh": [], "wo": [], "bo": 0.0}
    for j in range(H):
        rede["wh"].append([2 * r01() - 1, 2 * r01() - 1])
        rede["bh"].append(2 * r01() - 1)
        rede["wo"].append(2 * r01() - 1)
    rede["bo"] = 2 * r01() - 1
    return rede

def sigmoide(z):
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))
    ez = math.exp(z)
    return ez / (1.0 + ez)

def forward(rede, x):
    a = []
    for j in range(H):
        z = rede["bh"][j]
        for i in range(2):
            z += rede["wh"][j][i] * x[i]
        a.append(sigmoide(z))
    z = rede["bo"]
    for j in range(H):
        z += rede["wo"][j] * a[j]
    return a, sigmoide(z)

def perda(y, d):
    y = min(max(y, 1e-12), 1 - 1e-12)
    return -(d * math.log(y) + (1 - d) * math.log(1 - y))

def gradientes(rede, x, d):
    a, y = forward(rede, x)
    do = y - d
    dh = [do * rede["wo"][j] * a[j] * (1 - a[j]) for j in range(H)]   # pesos ANTES da atualização
    return {"wo": [do * a[j] for j in range(H)], "bo": do,
            "wh": [[dh[j] * x[0], dh[j] * x[1]] for j in range(H)], "bh": dh}, a, y

def passo(rede, x, d, eta):
    g, a, y = gradientes(rede, x, d)
    for j in range(H):
        rede["wo"][j] -= eta * g["wo"][j]
    rede["bo"] -= eta * g["bo"]
    for j in range(H):
        for i in range(2):
            rede["wh"][j][i] -= eta * g["wh"][j][i]
        rede["bh"][j] -= eta * g["bh"][j]
    return g

def perda_media(rede):
    return sum(perda(forward(rede, x)[1], d) for x, d in XOR) / 4

def vetor(rede):
    return [v for j in range(H) for v in rede["wh"][j]] + rede["bh"] + rede["wo"] + [rede["bo"]]

def de_vetor(v):
    return {"wh": [[v[2 * j], v[2 * j + 1]] for j in range(H)], "bh": v[8:12], "wo": v[12:16], "bo": v[16]}

# verificação numérica do gradiente (diferenças centrais)
r0 = criar_mlp()
pior = 0.0
for x, d in XOR:
    g = gradientes(r0, x, d)[0]
    gv = [v for j in range(H) for v in g["wh"][j]] + g["bh"] + g["wo"] + [g["bo"]]
    v = vetor(r0)
    for k in range(17):
        mais, menos = v.copy(), v.copy(); mais[k] += 1e-6; menos[k] -= 1e-6
        num = (perda(forward(de_vetor(mais), x)[1], d) - perda(forward(de_vetor(menos), x)[1], d)) / 2e-6
        pior = max(pior, abs(num - gv[k]))
print(f"Maior diferença entre gradiente analítico e numérico: {pior:.2e}")
assert pior < 1e-6

In [ ]:
rede = criar_mlp(); rede_inicial = json.loads(json.dumps(rede))
hist_perda, hist_grad, epoca_parada = [perda_media(rede)], [], None
for ep in range(1, 20001):
    norma = 0.0
    for x, d in XOR:
        g = passo(rede, x, d, 0.7)
        norma += sum(abs(v) for v in g["wo"]) + abs(g["bo"]) + sum(abs(v) for l in g["wh"] for v in l) + sum(abs(v) for v in g["bh"])
    hist_perda.append(perda_media(rede)); hist_grad.append(norma)
    if (ep % 100 == 0 or ep == 1) and hist_perda[-1] < 0.04:
        epoca_parada = ep
        break
print("Critério BCE < 0,04 atingido na época", epoca_parada)
probs = [forward(rede, x)[1] for x, _ in XOR]
for (x, d), p in zip(XOR, probs):
    print(f"x={x} y={p:.4f} classe={int(p >= 0.5)} esperado={d}")
if exec_mlp is not None:
    import re
    assert int(re.search(r"epoca=(\d+)\n", exec_mlp.stdout).group(1)) == epoca_parada
    assert re.findall(r"prob=(\d\.\d{4})", exec_mlp.stdout) == [f"{p:.4f}" for p in probs]
    print("Python reproduz o C: mesma época de parada e mesmas saídas (4 casas).")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(hist_perda, color="#b42318"); ax[0].axhline(0.04, ls=":", color="k", label="critério 0,04")
ax[0].set(title="Perda média por época", xlabel="época", ylabel="entropia cruzada média (nats)"); ax[0].legend()
ax[1].semilogy(range(1, len(hist_grad) + 1), hist_grad, color="#7c3aed")
ax[1].set(title="Soma de |gradiente| por época", xlabel="época", ylabel="Σ|∂L/∂θ| (escala log)")
A = np.array([forward(rede, x)[0] for x, _ in XOR])
im = ax[2].imshow(A, cmap="Blues", vmin=0, vmax=1)
ax[2].set(title="Ativações ocultas após o treino", xticks=range(4), xticklabels=["a₁", "a₂", "a₃", "a₄"],
          yticks=range(4), yticklabels=[f"x={x}, d={d}" for x, d in XOR])
for i in range(4):
    for j in range(4):
        ax[2].text(j, i, f"{A[i, j]:.2f}", ha="center", va="center", color="white" if A[i, j] > 0.6 else "black")
fig.colorbar(im, ax=ax[2], shrink=0.8, label="aⱼ")
plt.tight_layout(); plt.show()

g = np.linspace(-0.25, 1.25, 120)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
for k, (r, titulo) in enumerate([(rede_inicial, "Antes do treino"), (rede, f"Após {epoca_parada} épocas")]):
    Z = np.array([[forward(r, (a, b))[1] for a in g] for b in g])
    c = ax[k].contourf(g, g, Z, levels=np.linspace(0, 1, 11), cmap="BrBG")
    ax[k].contour(g, g, Z, levels=[0.5], colors="k", linewidths=2)
    for x, d in XOR:
        ax[k].scatter(*x, s=150, c="#146e83" if d else "white", edgecolors="k", zorder=3)
    ax[k].set(title=titulo + " — saída y(x₁, x₂)", xlabel="x₁", ylabel="x₂")
fig.colorbar(c, ax=ax, shrink=0.85, label="y (estimativa da classe 1)")
plt.show()
print("A linha preta (y = 0,5) deixa de ser uma reta: a camada oculta combina várias fronteiras.")

## 3. Experimento guiado B — MLP: mesma inicialização, duas taxas

**Pergunta:** o efeito da taxa de aprendizagem é igual ao observado no perceptron?

A célula abaixo parte da **mesma rede inicial** e executa o mesmo número de épocas para cada taxa, sem modificar a MLP original usada nos testes do C. Alterar somente `ETA_MLP_ALTERADA` permite comparar a velocidade da queda da perda.

**Atenção:** a perda é medida nos próprios quatro exemplos XOR usados para treinar. Ela **não mede generalização**. Preveja primeiro o resultado, execute e depois compare as curvas.

Inclua a curva e a interpretação no **relatório final único do Moodle**.


In [ ]:
ETA_MLP_ORIGINAL, ETA_MLP_ALTERADA = 0.7, 0.35  # Mude somente ETA_MLP_ALTERADA
EPOCAS_COMPARACAO = 400
assert 0 < ETA_MLP_ALTERADA <= 2
def percurso_mlp(eta):
    rede_exp = json.loads(json.dumps(rede_inicial))  # mesmos pesos iniciais
    perdas = [perda_media(rede_exp)]
    for _ in range(EPOCAS_COMPARACAO):
        for xx, dd in XOR:
            passo(rede_exp, xx, dd, eta)
        perdas.append(perda_media(rede_exp))
    return np.asarray(perdas)
perda_original = percurso_mlp(ETA_MLP_ORIGINAL)
perda_alterada = percurso_mlp(ETA_MLP_ALTERADA)
print(f"ANTES × DEPOIS — MLP em {EPOCAS_COMPARACAO} épocas, mesma inicialização")
print(f"η={ETA_MLP_ORIGINAL:g}: BCE final={perda_original[-1]:.4f}")
print(f"η={ETA_MLP_ALTERADA:g}: BCE final={perda_alterada[-1]:.4f}")
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(perda_original, label=f"original η={ETA_MLP_ORIGINAL:g}")
ax.plot(perda_alterada, label=f"alterado η={ETA_MLP_ALTERADA:g}")
ax.axhline(0.04, ls=":", color="gray", label="referência BCE 0,04")
ax.set(xlabel="época", ylabel="perda BCE (treino)", title="MLP XOR: duas taxas com os mesmos pesos iniciais")
ax.legend(); plt.tight_layout(); plt.show()
print("\nINTERPRETAÇÃO AUTOMÁTICA:")
if abs(perda_alterada[-1] - perda_original[-1]) < 1e-4:
    print("As perdas finais são praticamente iguais no número de épocas escolhido.")
elif perda_alterada[-1] < perda_original[-1]:
    print("A taxa alterada reduziu mais a perda de treino neste intervalo de épocas.")
else:
    print("A taxa alterada terminou com perda de treino maior neste intervalo de épocas.")
print("Isso é aprendizado no XOR, não evidência de melhor desempenho em dados externos.")
print("Registre a taxa escolhida, a comparação das curvas e uma limitação no relatório do Moodle.")



## Questões de interpretação

Explique por que a camada oculta resolve XOR e por que uma perda menor não mede generalização. Registre cálculo de referência, curvas de perda antes × depois e limitação no relatório único do Moodle.

## Conclusão

Este experimento é independente. Use o mesmo conjunto de dados e as mesmas condições iniciais ao comparar parâmetros. **Experimentos propostos:** altere apenas o valor indicado no código e justifique o resultado observado, sem afirmar melhora quando as métricas não a demonstram. Entregue somente um **único relatório final** pelo Moodle, conforme as instruções da disciplina.
